# Planejamento e Reflexão

Planejamento e reflexão resolvem problemas diferentes com a mesma moeda: gastar mais chamadas ao modelo antes de entregar a resposta. Planejar é escrever a sequência de passos antes de executar, em vez de decidir um passo por vez. Refletir é submeter a resposta a uma crítica e reescrevê-la. As duas técnicas custam tokens e latência, e nenhuma delas melhora tudo.

O notebook parte de uma tarefa que o laço de ferramentas não resolve, representa o plano como estrutura de dados validada, executa o plano passo a passo, replaneja depois de uma falha provocada, e depois trata a crítica com rubrica e o laço de revisão, incluindo o caso em que revisar não ajuda. A última parte compara as três rotas sob o mesmo orçamento de chamadas.

In [ ]:
# No Google Colab, descomente e rode uma vez (Ambiente de execução > GPU).
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

import time
from pathlib import Path
from typing import Literal

import pandas as pd
import torch
from pydantic import BaseModel, Field

from agentkit import LLM, run_agent, tool

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(MODEL_NAME, device=device, temperature=0.0, max_tokens=250)
print(llm.model)

## A tarefa que falha em um passo

Os três arquivos abaixo simulam relatórios mensais, cada um com um número dentro. A tarefa é somar os três, e ela só termina depois de listar a pasta, ler cada arquivo e fazer a conta.

In [ ]:
REPORTS = Path("workspace/reports")
REPORTS.mkdir(parents=True, exist_ok=True)
MONTHS = {"janeiro.txt": 1200, "fevereiro.txt": 950, "marco.txt": 1430}
for name, value in MONTHS.items():
    (REPORTS / name).write_text(f"Relatorio mensal\nTotal de entregas: {value}\n", encoding="utf-8")
print(sorted(path.name for path in REPORTS.iterdir()), "soma correta:", sum(MONTHS.values()))

In [ ]:
@tool
def list_files() -> str:
    """List the files available in the reports folder."""
    return ", ".join(sorted(path.name for path in REPORTS.iterdir()))


@tool
def read_file(name: str) -> str:
    """Read a file from the reports folder and return its content."""
    return (REPORTS / name).read_text(encoding="utf-8")


TOOLS = [list_files, read_file]

O laço de ferramentas do `agentkit` recebe a tarefa inteira. Antes de rodar, tente prever quantos passos ele vai dar.

In [ ]:
TASK = "Read every file in the reports folder, take the delivery total in each one, and sum them."
direct = run_agent(llm, TASK, tools=TOOLS, max_steps=6)
print(direct["answer"])
print(direct["stop_reason"], direct["usage"])

O laço inventou nomes de arquivo, escreveu duas chamadas de ferramenta em texto corrido e encerrou no primeiro passo, porque um texto com duas chamadas não é uma chamada válida e o laço o tratou como resposta final. Nenhuma ferramenta foi executada e nenhum número saiu de um arquivo.

A causa é estrutural. O laço decide o próximo passo olhando o histórico, e nada nele representa o objetivo, o que já foi feito e o que falta. Uma tarefa com vários passos e dependência entre eles não sobrevive a isso.

## Plano como estrutura

O plano é um objeto validado, e não um texto. Escrito como estrutura, ele pode ser impresso antes da execução, conferido, alterado por código e usado como registro do que já rodou.

### O esquema do plano

In [ ]:
class Step(BaseModel):
    id: int
    instruction: str


class Plan(BaseModel):
    goal: str
    steps: list[Step]

In [ ]:
def make_plan(task: str, tools: list, max_steps: int = 5) -> Plan:
    """Pede ao modelo um plano validado pelo esquema, com um passo por chamada de ferramenta."""
    available = ", ".join(fn.tool_schema["name"] for fn in tools)
    return llm.generate_structured([{"role": "user", "content": (
        f"Break the task into at most {max_steps} steps. "
        "Each step must be one instruction that a single tool call can satisfy.\n\n"
        f"Available tools: {available}\nTask: {task}"
    )}], Plan, max_tokens=300)

In [ ]:
plan = make_plan(TASK, TOOLS)
pd.DataFrame([step.model_dump() for step in plan.steps])

O plano existe antes de qualquer execução e pode ser lido por quem escreveu o programa. Essa é a diferença prática entre plano como estrutura e plano como parágrafo.

Ele também está errado. O passo do meio manda ler cada arquivo, o que esconde uma repetição dentro de um passo que o executor resolve com uma chamada só. Planejar é uma tarefa como qualquer outra e o modelo erra nela: o número de passos depende de um dado que ninguém observou ainda, que é a quantidade de arquivos na pasta.

### Plano derivado de uma observação

Como o plano é uma estrutura, ele pode ser construído por código a partir do que a execução observou. A listagem da pasta vem primeiro, e o plano nasce dela com um passo por arquivo.

In [ ]:
def plan_from_files(task: str) -> Plan:
    """Monta o plano a partir da listagem da pasta, com um passo por arquivo."""
    names = [name.strip() for name in list_files().split(",")]
    steps = [
        Step(id=index, instruction=f"Read the file {name} and report the delivery total.")
        for index, name in enumerate(names, start=1)
    ]
    return Plan(goal=task, steps=steps)

In [ ]:
grounded = plan_from_files(TASK)
pd.DataFrame([step.model_dump() for step in grounded.steps])

Cada passo agora é uma instrução que uma chamada de ferramenta resolve, e o número de passos veio do mundo em vez do palpite do modelo. A divisão de responsabilidade é a decisão de projeto desta parte: o modelo decide o que fazer, o código decide quantas vezes.

### Execução do plano

Cada passo vira uma chamada ao mesmo laço de ferramentas do `agentkit`. O executor não mudou; mudou o tamanho do problema que chega até ele.

In [ ]:
def execute_plan(plan: Plan, tools: list) -> list[dict]:
    """Executa os passos em ordem e devolve o resultado de cada um."""
    results = []
    for step in plan.steps:
        # Cada passo roda isolado. Passar os resultados anteriores como contexto
        # faz o modelo responder sobre o passo errado.
        outcome = run_agent(llm, step.instruction, tools=tools, max_steps=4)
        results.append({
            "id": step.id,
            "instruction": step.instruction,
            "result": (outcome["answer"] or "").strip(),
            "calls": outcome["usage"]["steps"],
        })
    return results

In [ ]:
results = execute_plan(grounded, TOOLS)
pd.DataFrame(results)[["id", "result"]]

Os três arquivos foram lidos e cada número saiu do arquivo certo. O que o laço não resolvia como tarefa única passou a ser resolvido como três tarefas pequenas.

### Agregação

Falta somar, e somar é onde este modelo continua ruim. A agregação não precisa de raciocínio: precisa de valores tipados, que é o problema da saída estruturada.

In [ ]:
class Totals(BaseModel):
    values: list[int]

In [ ]:
def aggregate(results: list[dict]) -> int:
    """Extrai um número por passo executado e soma em Python."""
    notes = "\n".join(f"Step {item['id']}: {item['result']}" for item in results)
    totals = llm.generate_structured([{"role": "user", "content": (
        f"Extract one delivery total per note. There are {len(results)} notes, "
        f"so return {len(results)} numbers.\n\n{notes}"
    )}], Totals, max_tokens=120)
    return sum(totals.values)

In [ ]:
total = aggregate(results)
print(total, "| correto:", sum(MONTHS.values()))

A soma sai de Python sobre inteiros validados, e o modelo fica com a parte que só ele faz, que é achar o número dentro de uma frase. Tirar a aritmética do modelo economiza pouco token e remove do caminho a etapa em que ele erra.

## Replanejamento

Plano fixo pressupõe que o mundo não muda durante a execução. A célula seguinte remove um dos arquivos.

In [ ]:
(REPORTS / "marco.txt").unlink()
print(list_files())

In [ ]:
stale = execute_plan(grounded, TOOLS)
pd.DataFrame(stale)[["id", "result"]]

O passo do arquivo removido falhou, a exceção voltou ao laço como observação e o modelo relatou a falha em vez de inventar um número. O plano continua o mesmo, porque foi montado antes da mudança.

Replanejar é derivar o plano de novo a partir do estado atual. Como este plano vem de uma observação, replanejar é repetir a observação.

In [ ]:
replanned = plan_from_files(TASK)
results = execute_plan(replanned, TOOLS)
print(aggregate(results), "| correto agora:", sum(MONTHS.values()) - MONTHS["marco.txt"])

In [ ]:
for name, value in MONTHS.items():
    (REPORTS / name).write_text(f"Relatorio mensal\nTotal de entregas: {value}\n", encoding="utf-8")
print(list_files())

O replanejamento custa a observação e a montagem do plano, e só se justifica quando a execução encontra algo que o plano não previa. Replanejar a cada passo gasta uma chamada por passo e converge para o laço intercalado que existia antes do plano.

## Reflexão

A segunda técnica não mexe no caminho até a resposta, e sim na resposta pronta.

### Crítica com rubrica

Pedir ao modelo que avalie um texto sem critério devolve elogio. A crítica útil tem rubrica declarada, saída estruturada e um veredito de um conjunto fechado.

In [ ]:
class Critique(BaseModel):
    score: int = Field(ge=0, le=5)
    issues: list[str]
    verdict: Literal["accept", "revise"]


def critique(task: str, answer: str) -> Critique:
    """Avalia uma resposta segundo a rubrica de precisão e completude."""
    return llm.generate_structured([{"role": "user", "content": (
        f"Task: {task}\nAnswer: {answer}\n\n"
        "Grade the answer from 0 to 5 for precision and completeness, "
        "list the concrete problems, and decide accept or revise."
    )}], Critique, max_tokens=250)

In [ ]:
weak_answer = "The sum of deliveries is around three thousand, more or less."
review = critique(TASK, weak_answer)
print(review.score, review.verdict)
for issue in review.issues:
    print(" -", issue)

A nota e a lista de problemas são valores, não prosa, e por isso o laço de revisão pode decidir com eles.

### Laço de revisão

In [ ]:
def revise(task: str, answer: str, max_rounds: int = 2) -> dict:
    """Critica e reescreve a resposta até o veredito de aceitação ou o fim das rodadas."""
    rounds = []
    for round_number in range(1, max_rounds + 1):
        review = critique(task, answer)
        rounds.append({"round": round_number, "score": review.score, "verdict": review.verdict})
        if review.verdict == "accept":
            break
        answer = llm.invoke([{"role": "user", "content": (
            f"Task: {task}\nPrevious answer: {answer}\n"
            f"Problems found: {review.issues}\n\nWrite an improved answer."
        )}], max_tokens=150)
    return {"answer": answer, "rounds": rounds}

In [ ]:
revised = revise(TASK, weak_answer)
print(revised["answer"])
pd.DataFrame(revised["rounds"])

A resposta mudou de forma e continua sem os números certos, porque nenhuma das chamadas do laço de revisão tem acesso aos arquivos. Reflexão melhora o que já está no contexto e não substitui a informação que falta: uma resposta errada por falta de dado continua errada depois de qualquer número de rodadas.

### Quando a revisão não ajuda

O caso seguinte parte de uma resposta correta e simples.

In [ ]:
question = "What is the capital of Australia? Answer with the city only."
first_answer = llm.invoke([{"role": "user", "content": question}], max_tokens=20)
print(first_answer)

In [ ]:
second_review = critique(question, first_answer)
print(second_review.score, second_review.verdict)
for issue in second_review.issues:
    print(" -", issue)

O crítico deu nota intermediária a uma resposta certa e pediu revisão, com apontamentos que não apontam erro nenhum. O pedido de revisão é gratuito no sentido literal: ele custa chamadas e não tem defeito para corrigir. O risco simétrico, que é o crítico convencer o modelo a trocar a resposta certa por uma errada, é o que torna obrigatório medir antes de adotar a técnica.

Um crítico que só aceita quando não encontra problema encontra problema sempre, porque encontrar problema é a tarefa que recebeu.

## Orçamento pareado

Comparar resposta direta com plano e com revisão só é honesto sob o mesmo orçamento de chamadas. A rota da autoconsistência gasta as chamadas amostrando várias respostas e ficando com a mais frequente.

In [ ]:
def sample_answers(task: str, tools: list, samples: int = 3) -> list[str]:
    """Roda o laço várias vezes com amostragem e devolve as respostas finais."""
    llm.temperature = 0.8
    answers = []
    for seed in range(samples):
        torch.manual_seed(seed)
        answers.append((run_agent(llm, task, tools=tools, max_steps=6)["answer"] or "").strip())
    llm.temperature = 0.0
    return answers

In [ ]:
started = time.perf_counter()
sampled = sample_answers(TASK, TOOLS)
sampling_seconds = time.perf_counter() - started
for answer in sampled:
    print("-", answer[:90].replace("\n", " "))

In [ ]:
started = time.perf_counter()
planned = execute_plan(plan_from_files(TASK), TOOLS)
planned_total = aggregate(planned)
planning_seconds = time.perf_counter() - started
print(planned_total)

In [ ]:
target = str(sum(MONTHS.values()))
pd.DataFrame([
    {"rota": "resposta direta", "chamadas": direct["usage"]["steps"],
     "segundos": direct["usage"]["seconds"], "acertou": target in (direct["answer"] or "")},
    {"rota": "autoconsistência", "chamadas": len(sampled),
     "segundos": round(sampling_seconds, 1), "acertou": any(target in answer for answer in sampled)},
    {"rota": "plano e execução", "chamadas": len(planned) + 1,
     "segundos": round(planning_seconds, 1), "acertou": planned_total == sum(MONTHS.values())},
])

A tabela é o fechamento da aula. As três amostras erram, cada uma de um jeito, e nenhuma votação salva um conjunto em que a resposta certa não aparece: amostrar mais vezes a mesma rota ruim multiplica o custo sem tocar na causa. O plano acerta com um número parecido de chamadas e em uma fração do tempo, porque muda a estrutura do problema em vez de gastar mais.

A conclusão que transfere para o resto do semestre é o critério de escolha: decomposição quando a tarefa tem passos com dependência, revisão quando existe rubrica objetiva e a informação necessária já está no contexto, e resposta direta quando nenhuma das duas condições vale.

## Exercícios

### Exercício 1

Escreva uma tarefa de três passos sobre os arquivos da pasta, como encontrar o mês de maior total, e rode as duas rotas: o laço direto e o plano executado. Compare as respostas e o número de chamadas.

In [ ]:
new_task = ""

### Exercício 2

Peça um plano com no máximo dois passos para a mesma tarefa da soma e execute. Diga em que ponto a decomposição deixou de ser suficiente.

In [ ]:
short_plan = None

### Exercício 3

Acrescente ao esquema `Step` um campo com a ferramenta esperada para aquele passo e compare, depois da execução, a ferramenta prevista com a que foi de fato chamada. Aponte os passos em que o plano errou a previsão.

In [ ]:
class TypedStep(BaseModel):
    ...

### Exercício 4

Reescreva a rubrica de `critique` para que ela exija evidência do arquivo em cada apontamento, e rode de novo sobre a resposta correta da capital. Verifique se o veredito muda de revisar para aceitar.

In [ ]:
STRICT_RUBRIC = ""

### Exercício 5

Rode `revise` com três rodadas sobre uma resposta ruim e registre a nota de cada rodada. Diga se a nota subiu de forma consistente e quantas chamadas foram gastas para chegar ao veredito final.

In [ ]:
rounds_budget = 3